# Data Foundation — worked solution

**Instructor copy. Do not hand out.** It names every defect on sight; a student who
reads this first has nothing left to discover.

Defects planted by reality, in the order students meet them:

| # | Defect | Where it bites |
|---|---|---|
| 1 | ~22% duplicate rows, up to hundreds of copies of one reading | Ex 2 — inflates any count or weighted mean |
| 2 | Weather station registered 3x, one per hive | Ex 3 — triples rows on a careless join, silently |
| 3 | Irregular, non-aligned sampling | Ex 4 — nothing joins on exact timestamps |
| 4 | No NULLs, but 3-6% of hours absent | Ex 4 — invisible until a complete index exists |
| 5 | UTC in a naive timestamp column | Ex 5 — local-time features come out shifted |

**This notebook reads live.** Students now extract a rolling three-month window
straight from Postgres rather than a frozen snapshot, so every number below moves
between runs, and the exact figures in the commentary are illustrative. Two
consequences worth knowing before you teach it:

- The duplicate-row defect is a **live bug in the `ChatBotCollector` service**. If it is
  ever fixed upstream, Exercise 2 goes quiet and the day loses its best lesson. Run this
  notebook the week before, and check.
- A three-month window may or may not contain a DST change. The Exercise 5 lesson holds
  either way — a naive UTC column still needs localising — but if the window is entirely
  inside summer time, nobody will *see* the offset move. Check before promising they will.

Caveats that are real but invisible in the data, worth mentioning aloud:

- The hive-to-sensor mapping is **unconfirmed**. `hives.toml` marks it `UNBESTÄTIGT`,
  and the database's mapping disagrees with that guess.
- `rainGauge` looks cumulative rather than per-interval, and is unverified against the
  source API.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

REPO = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "workshop" / "out"
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO / ".env")
DATABASE_URL = os.environ["DATABASE_URL"]
print("database:", DATABASE_URL.rsplit("@", 1)[-1])

## Exercise 0 — Extract

In [ ]:
import psycopg

# Three months. The filter belongs in SQL: `data` is millions of rows and all but
# the last few hundred thousand are outside the window.
# ts is `timestamp without time zone` holding UTC, so compare against UTC
# rather than the server's local clock.
queries = {
    "beehives": "SELECT * FROM beehives",
    "sensors":  "SELECT * FROM sensors",
    "data":     "SELECT * FROM data WHERE ts >= (now() AT TIME ZONE 'UTC') - INTERVAL '3 months'",
}

frames = {}  # table name -> its dataframe
with psycopg.connect(DATABASE_URL, connect_timeout=30) as conn:
    for name, sql in queries.items():
        with conn.cursor() as cur:
            cur.execute(sql)
            frames[name] = pd.DataFrame(cur.fetchall(),
                                        columns=[c.name for c in cur.description])

# Parquet, because dtypes and timestamps survive the round trip. CSV would
# hand `ts` and `value` back as strings and quietly cost them Exercise 4.
for name, df in frames.items():
    df.to_parquet(RAW / f"{name}.parquet", index=False)
    print(f"  {name:9} {len(df):>9,} rows -> {name}.parquet")

In [ ]:
# Read back from disk. Everything downstream works from these, so the rest of the
# day is reproducible and needs no network.
beehives = pd.read_parquet(RAW / "beehives.parquet")
sensors  = pd.read_parquet(RAW / "sensors.parquet")
data     = pd.read_parquet(RAW / "data.parquet")

print(f"beehives {beehives.shape}   sensors {sensors.shape}   data {data.shape}")
print("period:", data["ts"].min(), "->", data["ts"].max())

# The window we asked for, not the window any one device happened to fill.
# Exercise 4 reindexes onto this, so a hive silent for its last week still shows
# as absent. The END comes from the data, not the clock: if the whole collector
# has stalled, padding the grid out to now() invents an outage that is really
# just "the database stops here", and the validator would fail an honest table.
WINDOW_START = pd.Timestamp.now("UTC").tz_convert(None).floor("h") - pd.DateOffset(months=3)
WINDOW_END = data["ts"].max().floor("h")
print("requested from:", WINDOW_START)
print("grid:          ", max(WINDOW_START, data["ts"].min()).floor("h"), "->", WINDOW_END)

The common mistakes here are re-querying inside every later cell — turning a
half-day workshop into a load test — and saving to CSV, which round-trips `ts` to a
string and makes `.dt.floor("h")` fail two exercises later with a message that does
not mention CSV.

Someone usually asks why `beehives` and `sensors` are pulled whole. Twelve rows
between them: the cost of thinking about it exceeds the cost of fetching it.

## Exercise 1 — Profile

In [ ]:
# first rows: one measurement per row
data.head()

In [ ]:
data.dtypes

In [ ]:
data["measurement_unit"].value_counts()

In [ ]:
data["ts"].min(), data["ts"].max()

In [ ]:
# zero. Nothing is missing?
data["value"].isna().sum()

In [ ]:
# nine rows, seven physical devices — first hint of defect 2
sensors

In [ ]:
beehives

`data` is long: one row per *measurement*, not per device reading. Nine sensor rows
for seven physical devices — the first hint of defect 2, if anyone reads carefully.

Zero NULLs is the trap in Exercise 4: absence here is a missing *row*, not a missing value.

## Exercise 2 — Deduplicate

In [ ]:
KEY = ["sensor_id", "measurement_unit", "ts"]

before = len(data)
distinct = len(data.drop_duplicates(subset=KEY))
print(f"rows {before:,} -> distinct keys {distinct:,}  ({100*(1-distinct/before):.1f}% redundant)")

multiplicity = data.groupby(KEY, observed=True).size()
print("worst multiplicity:", int(multiplicity.max()), "copies of a single reading")

# Do the copies disagree? This decides drop-vs-arbitrate.
conflicts = int((data.groupby(KEY, observed=True)["value"].nunique() > 1).sum())
print("keys with conflicting values:", conflicts, "-> exact copies, safe to drop")

# reading_id proves these are separate INSERTs, not a query artifact.
worst = multiplicity.idxmax()
sample = data[(data["sensor_id"] == worst[0]) & (data["measurement_unit"] == worst[1])
              & (data["ts"] == worst[2])]
print("\ndistinct reading_ids for that one reading:", sample["reading_id"].nunique())

data = data.drop_duplicates(subset=KEY)
print("after dedup:", f"{len(data):,}")

An upstream collector bug, still live and running at 20–28% every month observed. Note
it is *uneven* across months, so it skews any mean weighted by row count — dropping it
is not merely cosmetic.

If this cell ever prints a redundancy near zero, the bug was fixed upstream between
workshops. Say so rather than bluffing; the lesson survives as "here is how you would
have caught it".

## Exercise 3 — Reshape and join

In [ ]:
ROLE = {
    "LoRaWAN Dragino-S31-LB": "food_chamber",
    "LoRaWAN Dragino-D23-LB": "brood_chamber",
    "LoRaWAN SenseCAP-S2120": "weather",
}
sensors = sensors.assign(role=sensors["sensor_type"].map(ROLE))

# THE TRAP: one physical weather station, registered once per hive.
weather_reg = sensors[sensors["role"] == "weather"]
print("weather registrations:", len(weather_reg),
      "| distinct devices:", weather_reg["sensor_name"].nunique())
print(weather_reg[["sensor_id", "beehive_id", "sensor_name"]].to_string(index=False))

In [ ]:
before = len(data)
data = data.merge(sensors[["sensor_id", "beehive_id", "role"]],
                  on="sensor_id", how="left", suffixes=("", "_s"))
print(f"data {before:,} -> {len(data):,} after merge")

In [ ]:
hive_rows = data[data["role"] != "weather"]
weather_rows = data[data["role"] == "weather"]
print(f"hive_rows    {len(hive_rows):>9,}")
print(f"weather_rows {len(weather_rows):>9,}  <- 3x inflated")

# Collapse to the single physical station. Values are identical across the three
# registrations, so dropping duplicate (unit, ts) pairs is exact, not an average.
weather_rows = weather_rows.drop_duplicates(subset=["measurement_unit", "ts"])
print(f"weather_rows {len(weather_rows):>9,}  <- collapsed")

A student who skips the collapse and merges ambient readings onto hive readings gets
three times the rows and no error. The validator's `weather shared across hives` check
exists for exactly this.

## Exercise 4 — Align time and handle gaps

In [ ]:
# 1. one clock: every ts belongs to an hour
hive_rows = hive_rows.assign(hour=hive_rows["ts"].dt.floor("h"))
print(hive_rows[["ts", "hour"]].head())
print("shape:", hive_rows.shape)

In [ ]:
# 2. still long: one row per hive-hour-measurement
hive_h = hive_rows.groupby(["beehive_id", "hour", "measurement_unit"], observed=True)["value"].mean()
print(hive_h.head())
print("index names:", hive_h.index.names, "  len:", f"{len(hive_h):,}")

In [ ]:
# 3. wide: each measurement becomes a column
hive_h = hive_h.unstack("measurement_unit")
print(hive_h.head())
print("hive_h wide:", hive_h.shape)

# weather is the same verbs, grain is (hour,) not (beehive_id, hour)
weather_rows = weather_rows.assign(hour=weather_rows["ts"].dt.floor("h"))
weather_h = (weather_rows.groupby(["hour", "measurement_unit"], observed=True)["value"]
             .mean().unstack("measurement_unit").add_prefix("outside_"))
print("weather_h:", weather_h.shape)

In [ ]:
# Complete grid from the extract window, not now() and not one device's coverage.
# A hive silent for its last week must still show as absent.
hours = pd.date_range(data["ts"].min().floor("h"), data["ts"].max().floor("h"), freq="h")
grid  = pd.MultiIndex.from_product(
    [beehives["beehive_id"], hours], names=["beehive_id", "hour"]
)
features = hive_h.reindex(grid)

print(f"observed hive-hours {len(hive_h):,} vs complete grid {len(features):,}"
      f"  -> {len(features) - len(hive_h):,} absent")
print("\nmissing % per column:")
print(features.isna().mean().mul(100).round(1).to_string())

In [ ]:
# How long are the gaps? The answer decides the policy.
gaps = (hive_h.reset_index().groupby("beehive_id")["hour"].diff()
        .dt.total_seconds().div(3600).dropna())
print("gap hours -- median {:.2f}, p99 {:.1f}, max {:.1f}".format(
    gaps.median(), gaps.quantile(0.99), gaps.max()))
print("gaps over 24h:", int((gaps > 24).sum()))

In [ ]:
# weather is per hour, not per hive
before = len(features)
features = features.join(weather_h, on="hour")
print(f"features {before:,} -> {len(features):,} after weather join")

# One defensible policy: interpolate gaps up to 3h; leave real outages empty
# rather than invent five days of readings. limit_area="inside" refuses to
# extrapolate off the ends.
MAX_GAP_H = 3
features = (features
            .groupby(level="beehive_id", group_keys=True)
            .apply(lambda g: g.droplevel("beehive_id")
                              .interpolate(method="time", limit=MAX_GAP_H,
                                           limit_area="inside")))
print("missing % after treatment:")
print(features.isna().mean().mul(100).round(1).to_string())

## Exercise 5 — Calendar, rename, validate

In [ ]:
features = features.reset_index()

# ts is UTC in a naive column. The hive is in Germany, so localise before extracting
# local-time features. Whether these three months cross a clock change depends on when
# you run this -- which is exactly why the offset is never hardcoded.
local = features["hour"].dt.tz_localize("UTC").dt.tz_convert("Europe/Berlin")
print("UTC offsets seen in this window:",
      sorted(local.dt.strftime("%z").unique()))

features["hour_local"] = local.dt.hour
features["month"] = local.dt.month
features["dayofweek"] = local.dt.dayofweek

In [ ]:
features = features.merge(beehives[["beehive_id", "name"]], on="beehive_id", how="left")
features = features.rename(columns={
    "name": "hive_name",
    "tempC1": "brood_temp_c1", "tempC2": "brood_temp_c2", "tempC3": "brood_temp_c3",
    "temperature": "food_temp", "relativeHumidity": "food_humidity",
    "outside_temperature": "outside_temp",
    "outside_relativeHumidity": "outside_humidity",
    "outside_windSpeed": "outside_wind_speed",
    "outside_windDirection": "outside_wind_dir",
    "outside_uvIndex": "outside_uv_index",
    "outside_pressure": "outside_pressure_pa",
    "outside_lightIntensity": "outside_light",
    "outside_rainGauge": "outside_rain",
})

assert not features.duplicated(subset=["beehive_id", "hour"]).any()
assert len(features) == features["beehive_id"].nunique() * features["hour"].nunique()

features.to_parquet(OUT / "features_hourly.parquet", index=False)
print(f"{len(features):,} rows x {features.shape[1]} cols -> {OUT / 'features_hourly.parquet'}")

```bash
uv run python workshop/validate_features.py workshop/out/features_hourly.parquet
```

Expected: all checks pass, with a few percent empty in the worst column — the multi-day
outages, correctly left as gaps rather than invented. The exact figure depends on which
three months you pulled; what matters is that it is neither 0% (outages invented or
dropped) nor above 25% (a join missed).

## Exercise 6 — Look at what you built

In [ ]:
import matplotlib.pyplot as plt

OPTIMAL_C = 35  # brood-nest target
DAYS = 7        # try 3; comment out the window line for the full extract

one = features[features["hive_name"] == "Beehive No.1"]
window = one[one["hour"] >= one["hour"].max() - pd.Timedelta(days=DAYS)]

ax = window.plot(x="hour", y=["brood_temp_c1", "outside_temp"], figsize=(10, 4))
ax.axhline(OPTIMAL_C, linestyle="--", label=f"optimal {OPTIMAL_C}°C")
ax.set_ylabel("°C")
ax.legend()

Three months of hourly points is too dense to read — that is why `DAYS` exists. Have them try `7` then `3`. Commenting out the window line is how they see the whole extract; it should look like a scribble.

The dashed line is 35°C, the brood-nest target, not a data series. Brood should hug that line more tightly than outside does. Breaks in the line are leftover gaps, not a plotting bug — pandas skips NaN. In a typical three-month window the longest hole is hours, not days.

If the two traces sit on top of each other, they plotted the same column twice or joined weather onto brood by mistake. If there is no dashed line, they never called `axhline`.